# WRAITH — GRPO Training Notebook

**WRAITH** (Weakness Recognition and Adaptive Intelligence for Tactical Hunting)  
trains a boss AI villain to study player behavioral patterns and exploit their weaknesses.

## Algorithm: Group Relative Policy Optimization (GRPO)
- For each player profile prompt, sample **G=8 completions** from the LLM
- Score each with the WRAITH multi-signal reward function
- Normalize within the group: `advantage_i = (r_i - mean) / std`
- Policy gradient update — **no value network needed**

## How to run
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Set your HuggingFace username + token in **Cell 7** before the final push

**Expected time:** ~25-35 min on Colab T4 for 500 episodes × 3 epochs

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl>=0.9.0" datasets accelerate bitsandbytes peft
!pip install fastapi pydantic openenv-core

## Cell 2 — Clone WRAITH repo (gets env.py, profiler.py, reward.py, combos.py)

In [ ]:
import os

if not os.path.exists("WRAITH"):
    !git clone https://github.com/V21-vani/WRAITH.git

%cd WRAITH
print("Working directory:", os.getcwd())
print("Files:", os.listdir("."))

## Cell 3 — Load base model with Unsloth + apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME      = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN     = 1024
NUM_GENERATIONS = 8      # G — completions sampled per prompt for GRPO
NUM_EPISODES    = 500    # synthetic training episodes
EPOCHS          = 3
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
LR              = 2e-5

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model + LoRA ready.")

## Cell 4 — Build training prompts from synthetic player episodes

We generate 500 synthetic fights with different player styles (left-heavy, right-heavy,
aggressive, mixed, panicking). Each produces a `profile_text` that becomes the LLM prompt.
The stored profile metadata is passed to the reward function later.

In [ ]:
import json
import random
from profiler import PlayerProfiler
from policy import SYSTEM_PROMPT

MOVE_POOL = ["DODGE_LEFT", "DODGE_RIGHT", "ATTACK"]
MOVE_WEIGHTS = {
    "left_heavy":  [0.70, 0.20, 0.10],
    "right_heavy": [0.20, 0.70, 0.10],
    "aggressive":  [0.20, 0.20, 0.60],
    "mixed":       [0.40, 0.40, 0.20],
    "panic_left":  [0.65, 0.15, 0.20],
}

def synthetic_episode(style: str, n_moves: int = 12) -> tuple:
    """Simulate a player with a given style; return profile text + metadata."""
    profiler = PlayerProfiler()
    weights  = MOVE_WEIGHTS.get(style, MOVE_WEIGHTS["mixed"])
    start_hp = random.uniform(15.0, 100.0)
    boss_hp  = random.uniform(20.0, 100.0)

    for i in range(n_moves):
        move   = random.choices(MOVE_POOL, weights=weights)[0]
        cur_hp = max(5.0, start_hp - i * (85.0 / max(n_moves, 1)))
        profiler.update(move, cur_hp)

    return (
        profiler.get_profile_text(),
        profiler.get_profile(),
        boss_hp,
        cur_hp,
    )

print(f"Generating {NUM_EPISODES} training prompts...")
records = []
styles  = list(MOVE_WEIGHTS.keys())

for _ in range(NUM_EPISODES):
    style = random.choice(styles)
    n     = random.randint(4, 18)
    profile_text, profile, boss_hp, player_hp = synthetic_episode(style, n)

    records.append({
        "prompt": tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": profile_text},
            ],
            tokenize=False,
            add_generation_prompt=True,
        ),
        "_profile":   json.dumps(profile),
        "_boss_hp":   boss_hp,
        "_player_hp": player_hp,
    })

from datasets import Dataset
dataset = Dataset.from_list(records)
print(f"Dataset ready: {len(dataset)} prompts")
print("\nSample profile text:")
print(records[0]["prompt"][:600])

## Cell 5 — Define the GRPO reward function

For each LLM completion the trainer generates, this function:
1. Parses the JSON `{"combo": ..., "reasoning": ...}` output
2. Rebuilds a minimal environment seeded with the stored player profile
3. Runs `env.step(action)` to get hit/win outcome
4. Calls `compute_reward()` which scores hit accuracy, exploit targeting, reasoning quality, and win

GRPO then normalizes these rewards within the group of 8 to compute per-completion advantages.

In [ ]:
import re
from env import WraithEnvironment
from models import WraithAction
from reward import compute_reward
from combos import COMBOS

VALID_COMBOS = set(COMBOS.keys())

def _parse(text: str) -> dict:
    """Extract combo + reasoning from raw LLM output."""
    try:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            d    = json.loads(m.group())
            name = d.get("combo", "PHANTOM_RUSH").upper().strip()
            if name not in VALID_COMBOS:
                name = "PHANTOM_RUSH"
            return {"combo": name, "reasoning": d.get("reasoning", text)}
    except Exception:
        pass
    return {"combo": "PHANTOM_RUSH", "reasoning": text}

def reward_fn(completions, prompts=None, **kwargs):
    """
    GRPO reward function.
    Called once per batch; returns one float reward per completion.
    """
    rewards    = []
    batch_meta = kwargs.get("batch", [{}] * len(completions))

    for i, text in enumerate(completions):
        parsed     = _parse(text)
        combo_name = parsed["combo"]
        reasoning  = parsed["reasoning"]

        meta      = batch_meta[i] if i < len(batch_meta) else {}
        profile   = json.loads(meta.get("_profile", "{}"))
        boss_hp   = float(meta.get("_boss_hp",   100.0))
        player_hp = float(meta.get("_player_hp", 100.0))

        # Rebuild env seeded with stored profile
        env = WraithEnvironment()
        env.state_data.boss_hp   = boss_hp
        env.state_data.player_hp = player_hp

        lb = profile.get("left_bias",  50) / 100.0
        rb = profile.get("right_bias", 50) / 100.0
        n  = max(profile.get("rounds", 4), 4)
        for _ in range(n):
            roll = random.random()
            if roll < lb:       env.profiler.update("DODGE_LEFT",  player_hp)
            elif roll < lb+rb:  env.profiler.update("DODGE_RIGHT", player_hp)
            else:               env.profiler.update("ATTACK",      player_hp)

        combo  = COMBOS.get(combo_name, COMBOS["PHANTOM_RUSH"])
        action = WraithAction(
            attack=combo_name,
            combo_name=combo_name,
            combo_threat=combo.threat_level,
            reasoning=reasoning,
        )

        obs = env.step(action)
        r, breakdown = compute_reward(
            action=action,
            profile=profile,
            hit=obs.metadata.get("hit", False),
            boss_won=obs.metadata.get("boss_won", False),
            combo=combo,
        )
        rewards.append(float(r))

    return rewards

# Quick sanity check
test_completions = [
    '{"combo": "WRATH_INCARNATE", "reasoning": "Player has dominant left dodge bias of 73%. WRATH_INCARNATE exploits left pattern with 82% hit probability. Confidence HIGH."}',
    '{"combo": "SHADOW_OBSERVER", "reasoning": "I will wait."}',
]
test_profile = json.dumps({"dominant_dodge": "LEFT", "left_bias": 73, "right_bias": 27,
                            "is_panicking": False, "rounds": 12, "confidence": "HIGH",
                            "attack_rate": 10, "total_dashes": 3})
test_meta = [{"_profile": test_profile, "_boss_hp": 80.0, "_player_hp": 60.0}] * 2
test_rewards = reward_fn(test_completions, batch=test_meta)
print(f"Sanity check rewards: {test_rewards}")
print(f"  Good combo (WRATH_INCARNATE): {test_rewards[0]:.2f}")
print(f"  Bad  combo (SHADOW_OBSERVER): {test_rewards[1]:.2f}")

## Cell 6 — Train with GRPO

The trainer samples 8 completions per prompt, scores each with `reward_fn`,
computes group-normalized advantages, and updates the policy.
Loss and reward curves are logged every 10 steps.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import matplotlib.pyplot as plt

training_args = GRPOConfig(
    output_dir="wraith-grpo-checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_new_tokens=220,
    temperature=0.85,
    learning_rate=LR,
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=100,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
train_result = trainer.train()
print("Training complete!")
print(f"Total steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")

## Cell 6b — Plot reward and loss curves

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps   = [x["step"]   for x in log_history if "loss" in x]
losses  = [x["loss"]   for x in log_history if "loss" in x]
rewards_log = [x.get("reward", None) for x in log_history if "loss" in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, losses, color='#cc2244', linewidth=1.5)
axes[0].set_title("GRPO Training Loss", fontsize=13)
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

if any(r is not None for r in rewards_log):
    clean_rewards = [r for r in rewards_log if r is not None]
    clean_steps   = [steps[i] for i, r in enumerate(rewards_log) if r is not None]
    axes[1].plot(clean_steps, clean_rewards, color='#2244cc', linewidth=1.5)
    axes[1].set_title("Mean Episode Reward per Step", fontsize=13)
    axes[1].set_xlabel("Training Step")
    axes[1].set_ylabel("Reward")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Reward not logged\n(add report_to=wandb for detailed curves)",
                 ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle("WRAITH GRPO Training — Qwen2.5-1.5B LoRA", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("wraith_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: wraith_training_curves.png")

## Cell 7 — Before/after comparison (baseline vs trained)

Run 50 episodes with a left-heavy player against:
- **Baseline**: untrained LLM (random combo selection)
- **Trained**: GRPO-finetuned WRAITH

Measure hit rate and mean reward to show the training worked.

In [ ]:
import numpy as np
from env import WraithEnvironment

def evaluate_policy(use_trained: bool, n_episodes: int = 50, style: str = "left_heavy") -> dict:
    """Run n_episodes and return hit_rate, mean_reward, win_rate."""
    if use_trained:
        FastLanguageModel.for_inference(model)

    hit_rates, episode_rewards, wins = [], [], []
    weights = MOVE_WEIGHTS[style]

    for ep in range(n_episodes):
        env = WraithEnvironment()
        obs = env.reset()
        ep_rewards, ep_hits = [], []

        for rnd in range(8):  # 8 rounds per episode
            # simulate 4 player moves before each WRAITH action
            for _ in range(4):
                move   = random.choices(MOVE_POOL, weights=weights)[0]
                cur_hp = env.state_data.player_hp
                env.profiler.update(move, cur_hp)

            profile_text = env.profiler.get_profile_text()
            profile      = env.profiler.get_profile()

            if use_trained:
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": profile_text},
                ]
                input_ids = tokenizer.apply_chat_template(
                    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
                ).to("cuda")
                with torch.no_grad():
                    out = model.generate(input_ids, max_new_tokens=180,
                                         do_sample=True, temperature=0.8)
                text   = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
                parsed = _parse(text)
            else:
                # baseline: random combo
                combo_name = random.choice(list(VALID_COMBOS))
                parsed     = {"combo": combo_name,
                              "reasoning": f"Randomly selected {combo_name}."}

            combo  = COMBOS.get(parsed["combo"], COMBOS["PHANTOM_RUSH"])
            action = WraithAction(
                attack=parsed["combo"],
                combo_name=parsed["combo"],
                combo_threat=combo.threat_level,
                reasoning=parsed["reasoning"],
            )
            obs = env.step(action)
            r, _ = compute_reward(
                action=action, profile=profile,
                hit=obs.metadata["hit"],
                boss_won=obs.metadata["boss_won"],
                combo=combo,
            )
            ep_rewards.append(r)
            ep_hits.append(int(obs.metadata["hit"]))
            if obs.done:
                break

        hit_rates.append(np.mean(ep_hits))
        episode_rewards.append(np.sum(ep_rewards))
        wins.append(int(env.state_data.player_hp <= 0))

    return {
        "hit_rate":    np.mean(hit_rates),
        "mean_reward": np.mean(episode_rewards),
        "win_rate":    np.mean(wins),
        "rewards":     episode_rewards,
    }

print("Evaluating baseline (random combos)...")
baseline = evaluate_policy(use_trained=False, n_episodes=50)

print("Evaluating trained WRAITH...")
trained  = evaluate_policy(use_trained=True,  n_episodes=50)

print("\n=== RESULTS (left-heavy player, 50 episodes) ===")
print(f"{'Metric':<20} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 52)
for key, label in [("hit_rate", "Hit Rate"), ("mean_reward", "Mean Reward"), ("win_rate", "Win Rate")]:
    b, t = baseline[key], trained[key]
    print(f"{label:<20} {b:>10.3f} {t:>10.3f} {t-b:>+10.3f}")

## Cell 7b — Plot baseline vs trained comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative reward over 50 episodes
axes[0].plot(np.cumsum(baseline["rewards"]), label="Baseline (random)",
             color='#888888', linewidth=1.5, linestyle='--')
axes[0].plot(np.cumsum(trained["rewards"]),  label="Trained WRAITH (GRPO)",
             color='#cc2244', linewidth=2.0)
axes[0].set_title("Cumulative Reward — Baseline vs Trained", fontsize=13)
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Cumulative Reward")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart of key metrics
metrics   = ["Hit Rate", "Win Rate"]
b_vals    = [baseline["hit_rate"], baseline["win_rate"]]
t_vals    = [trained["hit_rate"],  trained["win_rate"]]
x         = range(len(metrics))
width     = 0.35

axes[1].bar([xi - width/2 for xi in x], b_vals, width, label="Baseline", color='#888888')
axes[1].bar([xi + width/2 for xi in x], t_vals, width, label="Trained",  color='#cc2244')
axes[1].set_title("Hit Rate & Win Rate (50 episodes)", fontsize=13)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(metrics)
axes[1].set_ylabel("Rate")
axes[1].set_ylim(0, 1.0)
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle("WRAITH GRPO — Improvement Evidence", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("wraith_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: wraith_comparison.png")

## Cell 8 — Save LoRA + push to HuggingFace Hub

**Fill in your HF username and token before running.**  
Get a token at https://huggingface.co/settings/tokens (needs write access)

In [ ]:
HF_USERNAME = "YOUR_HF_USERNAME"   # ← replace
HF_TOKEN    = "hf_..."             # ← replace

model.save_pretrained("wraith-boss-ai-lora")
tokenizer.save_pretrained("wraith-boss-ai-lora")
print("Saved LoRA weights to wraith-boss-ai-lora/")

model.push_to_hub(f"{HF_USERNAME}/wraith-boss-ai", token=HF_TOKEN)
tokenizer.push_to_hub(f"{HF_USERNAME}/wraith-boss-ai", token=HF_TOKEN)

print(f"\nPushed to: https://huggingface.co/{HF_USERNAME}/wraith-boss-ai")
print("\nTo activate LLM in the game server:")
print(f"  WRAITH_USE_LLM=1 WRAITH_MODEL={HF_USERNAME}/wraith-boss-ai python app.py")